# Ad-level Threshold Tuning Notebook
- Load Ad-level table
- Explore distribution of key metrics
- Visualize natural clusters
- Define and tune thresholds for Local v National
- Keep Addressable as a separate concept

Key Features used:
- Average covereage (Weighted)
- Average Entropy (Weighted)
- Average KL (Weighted)
- Average Significant DMAs (Weighted)
- MAX Significant DMAs
- P-hat pooled
- Average Station Mix Ratios (Weighted)
- Average competitive Pressure


In [0]:
!pip install --upgrade matplotlib
%restart_python


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

In [0]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [0]:
df = spark.table("dev.mohit_gangwani.ad_labeling_ad_level_final_table_new").toPandas()

In [0]:
df.info()

In [0]:
df.rename(
    columns={
        "avg_coverage_weighted": "coverage_score",
        "avg_entropy_weighted": "entropy_norm",
        "avg_kl_weighted": "kl",
        "avg_sig_dma_weighted": "sig_dma_count",
        "avg_mix_ratio_station_weighted": "mix_ratio_station",
        "avg_competitive_pressure": "competitive_pressure",
    },
    inplace=True,
)

In [0]:
df.describe()

In [0]:
df.head(20)

In [0]:
metrics = [
    "p_hat_pooled",
    "coverage_score",
    "entropy_norm",
    "kl",
    "sig_dma_count",
    "avg_live_share_weighted",
    "mix_ratio_station",
    "competitive_pressure",
    "min_competitive_pressure",
    "max_competitive_pressure",
]
for col in metrics:
    plt.figure()
    plt.hist(df[col], bins=50)
    plt.yscale("log")
    plt.title(f"Distribution of {col.replace('_', ' ').title()}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [0]:
def basic_assertions(df: pd.DataFrame):
    # No missing must-have features
    must_have = [
        "coverage_score",
        "entropy_norm",
        "kl",
        "sig_dma_count",
        "competitive_pressure",
    ]
    for col in must_have:
        nulls = df[col].isna().sum()
        assert nulls == 0, f"{col} has {nulls} nulls - fix upstream."

    # Coverage in [0, 1]
    assert df["coverage_score"].between(0, 1).all(), "coverage_score out of [0,1]"

    # Entropy in [0, 1]
    assert df["entropy_norm"].between(0, 1).all(), "entropy_norm out of [0,1]"

    # KL non-negative
    assert (df["kl"] >= 0).all(), "kl has negative values"

    # Sig DMA non-negative
    assert (df["sig_dma_count"] >= 0).all(), "sig_dma_count has negative values"

    print("Basic assertions passed.")

In [0]:
basic_assertions(df)

In [0]:
# Initial thresholds – THESE ARE MEANT TO BE TUNED
thresholds = {
    # NATIONAL heuristics
    "nat_cov_min": 0.250,  # coverage_score ≥ 0.100
    "nat_entropy_min": 0.770,  # entropy_norm ≥ 0.770
    "nat_kl_max": 2.000,  # kl ≤ 2
    "nat_sig_dma_min": 5.000,  # sig_dma_count spread across many DMAs
    "nat_p_hat_cv_max": 0.50,  # p̂ fairly similar across DMAs (cv small)
    "nat_mix_ratio_tol": 0.30,  # |mix_ratio_station - 1| ≤ 0.30
    "nat_comp_min": 5.0,  # competitive_pressure high
    # LOCAL heuristics
    "loc_cov_max": 0.100,  # coverage_score ≤ 0.100
    "loc_entropy_max": 0.600,  # entropy_norm ≤ 0.600
    "loc_kl_min": 3.00,  # kl ≥ 3.0
    "loc_sig_dma_max": 3,  # sig_dma_count in only a few DMAs
    "loc_p_hat_cv_min": 0.80,  # p̂ strong in few DMAs, near-zero elsewhere
    "loc_mix_ratio_min": 1.50,  # mix_ratio_station skewed to specific stations
    "loc_comp_max": 3.0,  # competitive_pressure low
}

In [0]:
NAT_COLS = [
    "cond_nat_cov",
    "cond_nat_ent",
    "cond_nat_kl",
    "cond_nat_sig",
    "cond_nat_pcv",
    "cond_nat_mix",
    "cond_nat_comp",
]

LOC_COLS = [
    "cond_loc_cov",
    "cond_loc_ent",
    "cond_loc_kl",
    "cond_loc_sig",
    "cond_loc_pcv",
    "cond_loc_mix",
    "cond_loc_comp",
]

In [0]:
def add_condition_flags(df: pd.DataFrame, th: dict) -> pd.DataFrame:
    """
    Adds boolean columns for each national/local condition.
    """
    df = df.copy()

    # NATIONAL conditions
    df["cond_nat_cov"] = df["coverage_score"] >= th["nat_cov_min"]
    df["cond_nat_ent"] = df["entropy_norm"] >= th["nat_entropy_min"]
    df["cond_nat_kl"] = df["kl"] <= th["nat_kl_max"]
    df["cond_nat_sig"] = df["sig_dma_count"] >= th["nat_sig_dma_min"]
    df["cond_nat_pcv"] = df["p_hat_pooled"].fillna(np.inf) <= th["nat_p_hat_cv_max"]
    df["cond_nat_mix"] = (
        df["mix_ratio_station"].sub(1.0).abs() <= th["nat_mix_ratio_tol"]
    )
    df["cond_nat_comp"] = df["competitive_pressure"] >= th["nat_comp_min"]

    # LOCAL conditions
    df["cond_loc_cov"] = df["coverage_score"] <= th["loc_cov_max"]
    df["cond_loc_ent"] = df["entropy_norm"] <= th["loc_entropy_max"]
    df["cond_loc_kl"] = df["kl"] >= th["loc_kl_min"]
    df["cond_loc_sig"] = df["sig_dma_count"] <= th["loc_sig_dma_max"]
    df["cond_loc_pcv"] = df["p_hat_pooled"].fillna(0.0) >= th["loc_p_hat_cv_min"]
    df["cond_loc_mix"] = df["mix_ratio_station"] >= th["loc_mix_ratio_min"]
    df["cond_loc_comp"] = df["competitive_pressure"] <= th["loc_comp_max"]

    return df

In [0]:
def classify_row_conditions(row) -> str:
    nat_score = int(row[NAT_COLS].sum())
    loc_score = int(row[LOC_COLS].sum())

    nat_hit = nat_score >= 4
    loc_hit = loc_score >= 4

    # Clear cases
    if nat_hit and not loc_hit:
        return "National"
    if loc_hit and not nat_hit:
        return "Local"

    # Both hit: pick stronger score; tie -> mixed
    if nat_hit and loc_hit:
        if nat_score > loc_score:
            return "National"
        elif loc_score > nat_score:
            return "Local"
        else:
            return "Mixed"

    # Neither hit => mixed / unclassified
    return "Mixed"

In [0]:
def classify_ads(df_ads: pd.DataFrame, thresholds: dict) -> pd.DataFrame:
    """
    Main entry point:
      - adds condition flags
      - computes nat/local scores
      - returns df with `ad_type` and per-condition columns for debugging.
    """
    df = add_condition_flags(df_ads, thresholds)

    # scores
    df["nat_score"] = df[NAT_COLS].sum(axis=1)
    df["loc_score"] = df[LOC_COLS].sum(axis=1)

    # final label
    df["ad_type"] = df.apply(classify_row_conditions, axis=1)

    return df

In [0]:
new_df = classify_ads(df_ads=df, thresholds=thresholds)
new_df["ad_type"].value_counts(dropna=False)

In [0]:
def strict_assertions(
    df: pd.DataFrame, ad_type_col: str, max_local_cov: float = 0.65, min_national_cov: float = 0.25
):
    # No "local" ad should have VERY high coverage
    bad_local = df[(df[ad_type_col] == "Local") & (df["coverage_score"] > max_local_cov)]
    assert bad_local.empty, (
        f"{len(bad_local)} 'Local' ads have coverage > {max_local_cov}. "
        "Thresholds are probably too loose."
    )

    # No "national" ad should have very low coverage
    bad_nat = df[
        (df[ad_type_col] == "National") & (df["coverage_score"] < min_national_cov)
    ]
    assert bad_nat.empty, (
        f"{len(bad_nat)} 'National' ads have coverage < {min_national_cov}. "
        "Thresholds are probably too loose."
    )

    # No national ad with insane KL
    bad_nat_kl = df[(df[ad_type_col] == "National") & (df["kl"] > 1.5)]
    assert (
        bad_nat_kl.empty
    ), f"{len(bad_nat_kl)} 'National' ads have KL > 1.5 - likely misclassified."

    # No local ad with extremely low KL and very high entropy
    bad_loc_pattern = df[
        (df[ad_type_col] == "Local") & (df["kl"] < 0.1) & (df["entropy_norm"] > 0.8)
    ]
    assert (
        bad_loc_pattern.empty
    ), f"{len(bad_loc_pattern)} 'Local' ads look globally spread + low KL - check thresholds."

    print("Strict label assertions passed (no obviously impossible assignments).")

In [0]:
def inspect_label_sanity(df: pd.DataFrame, ad_type_col: str):
    # Quick group stats
    stats = df.groupby(ad_type_col)[
        [
            "coverage_score",
            "entropy_norm",
            "kl",
            "sig_dma_count",
            "p_hat_pooled",
            "mix_ratio_station",
            "competitive_pressure",
            "nat_score",
            "loc_score",
        ]
    ].agg(["mean", "median", "min", "max"])
    print("=== Per-label feature summary ===")
    print(stats)

    # Intuitive expectations:
    # 1) Nationals should have higher coverage than locals, on average
    cov_by_type = df.groupby(ad_type_col)["coverage_score"].mean().to_dict()
    if "National" in cov_by_type and "Local" in cov_by_type:
        assert (
            cov_by_type["National"] > cov_by_type["Local"]
        ), "Nationals don't have higher coverage than locals on average - check thresholds."

    # 2) Locals should have higher KL than nationals, on average
    kl_by_type = df.groupby(ad_type_col)["kl"].mean().to_dict()
    if "National" in kl_by_type and "Local" in kl_by_type:
        assert (
            kl_by_type["Local"] > kl_by_type["National"]
        ), "Locals don't have higher KL than nationals on average - check thresholds."

    # 3) Nationals should have higher competitive pressure than locals, on average
    # comp_by_type = df.groupby(ad_type_col)["competitive_pressure"].mean().to_dict()
    # if "National" in comp_by_type and "Local" in comp_by_type:
    #     assert (
    #         comp_by_type["National"] > comp_by_type["Local"]
    #     ), "Nationals don't have higher competitive pressure than locals - check thresholds."

    print("Label sanity checks passed (coverage, KL, comp align with intuition).")

In [0]:
strict_assertions(df=new_df, ad_type_col='ad_type')

In [0]:
inspect_label_sanity(df=new_df, ad_type_col='ad_type')

In [0]:
def get_seed_sets(df: pd.DataFrame, nat_min_score: int = 4, loc_min_score: int = 4):
    """
    High-confidence seeds:
      - national_seed: ads that strongly look national
      - local_seed: ads that strongly look local
    """
    nat_seed = df[df["nat_score"] >= nat_min_score].copy()
    loc_seed = df[df["loc_score"] >= loc_min_score].copy()
    return nat_seed, loc_seed

In [0]:
nat_seed, loc_seed = get_seed_sets(new_df, nat_min_score=4, loc_min_score=4)
len(nat_seed), len(loc_seed)

In [0]:
def print_seed_quantiles(nat_seed, loc_seed):
    cols = [
        "coverage_score",
        "entropy_norm",
        "kl",
        "sig_dma_count",
        "p_hat_pooled",
        "competitive_pressure",
    ]
    nat_seed_float = nat_seed[cols].astype(float)
    loc_seed_float = loc_seed[cols].astype(float)

    print("=== National seed quantiles ===")
    display(nat_seed_float.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))

    print("\n=== Local seed quantiles ===")
    display(loc_seed_float.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))

In [0]:
print_seed_quantiles(nat_seed, loc_seed)

In [0]:
def classify_v2_core(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    kl = row["kl"]
    sig = row["sig_dma_count"]

    # NATIONAL (proposed v2)
    nat_conds = [
        cov >= 0.0469,
        ent >= 0.8047,
        kl <= 2.690,
        sig >= 6.3,
    ]
    nat_score = sum(nat_conds)

    # LOCAL
    loc_conds = [
        cov <= 0.0145,
        ent <= 0.5532,
        kl >= 3.243,
        sig <= 2.2,
    ]
    loc_score = sum(loc_conds)

    if nat_score >= 3 and loc_score < 3:
        return "National"
    if loc_score >= 3 and nat_score < 3:
        return "Local"
    return "Mixed"

In [0]:
new_df["ad_type_v2"] = new_df.apply(classify_v2_core, axis=1)
nat = new_df[new_df["ad_type_v2"] == "National"]
loc = new_df[new_df["ad_type_v2"] == "Local"]
len(nat), len(loc)

In [0]:
def inspect_label_sanity(df):
    stats = df.groupby("ad_type_v2")[
        ["coverage_score", "entropy_norm", "kl", "sig_dma_count"]
    ].agg(["mean", "median", "min", "max"])
    display(stats)

    cov_by_type = df.groupby("ad_type_v2")["coverage_score"].mean().to_dict()
    kl_by_type = df.groupby("ad_type_v2")["kl"].mean().to_dict()

    if "National" in cov_by_type and "Local" in cov_by_type:
        assert (
            cov_by_type["National"] > cov_by_type["Local"]
        ), "Nationals don't have higher coverage than locals on average - thresholds off."

    if "National" in kl_by_type and "Local" in kl_by_type:
        assert (
            kl_by_type["Local"] > kl_by_type["National"]
        ), "Locals don't have higher KL than nationals - thresholds off."

    display("Label sanity checks (coverage & KL) passed.")

In [0]:
inspect_label_sanity(new_df)

In [0]:
def summarize_quantiles(df, label):
    print(f"=== {label} quantiles ===")
    display(
        df[["coverage_score", "entropy_norm", "kl", "sig_dma_count"]].astype(float).quantile(
            [0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]
        )
    )

In [0]:
summarize_quantiles(nat, "National")
summarize_quantiles(loc, "Local")

In [0]:
def classify_v3_core(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    kl = row["kl"]
    sig = row["sig_dma_count"]

    # NATIONAL (proposed v2)
    nat_conds = [
        cov >= 0.7141,
        ent >= 0.8458,
        kl <= 2.463,
        sig >= 7.7,
    ]
    nat_score = sum(nat_conds)

    # LOCAL
    loc_conds = [
        cov < 0.0108,
        ent < 0.4385,
        kl >= 3.288,
        sig <= 2.0,
    ]
    loc_score = sum(loc_conds)

    if nat_score >= 3 and loc_score < 3:
        return "National"
    if loc_score >= 3 and nat_score < 3:
        return "Local"
    return "Mixed"

In [0]:
new_df["ad_type_v3"] = new_df.apply(classify_v3_core, axis=1)

nat = new_df[new_df["ad_type_v3"] == "National"]
loc = new_df[new_df["ad_type_v3"] == "Local"]
len(nat), len(loc)

In [0]:
summarize_quantiles(nat, "National")
summarize_quantiles(loc, "Local")

In [0]:
def classify_binary_v1(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    kl = row["kl"]
    sig = row["sig_dma_count"]

    # ----- HARDCODED COVERAGE OVERRIDES -----
    # If extremely low coverage, always local
    if cov < 0.01:
        return "Local"

    # If sufficiently high coverage, always national
    if cov >= 0.10:
        return "National"

    # ----- NATIONAL CONDITIONS -----
    nat_conds = [cov >= 0.07, ent >= 0.8, kl <= 2, sig >= 6]
    nat_score = sum(nat_conds)

    # ----- LOCAL CONDITIONS -----
    loc_conds = [cov < 0.01, ent < 0.6, kl >= 3, sig <= 2]
    loc_score = sum(loc_conds)

    # ----- DECISION (NO MIXED) -----
    # Strong votes
    if nat_score >= 3:
        return "National"
    if loc_score >= 3:
        return "Local"

    # Weak votes → pick the closer cluster
    if nat_score > loc_score:
        return "National"
    if loc_score > nat_score:
        return "Local"

    # True 50/50 tie → use global bias toward national
    return "National"

In [0]:
new_df["ad_type_bin"] = new_df.apply(classify_binary_v1, axis=1)

nat = new_df[new_df["ad_type_bin"] == "National"]
loc = new_df[new_df["ad_type_bin"] == "Local"]
len(nat), len(loc)

In [0]:
summarize_quantiles(nat, "National")
summarize_quantiles(loc, "Local")

In [0]:
def propose_thresholds_from_seeds(nat_seed, loc_seed):
    # Coverage
    nat_cov_min = nat_seed["coverage_score"].astype(float).quantile(0.10)
    loc_cov_max = loc_seed["coverage_score"].astype(float).quantile(0.90)

    # Entropy
    nat_ent_min = nat_seed["entropy_norm"].astype(float).quantile(0.10)
    loc_ent_max = loc_seed["entropy_norm"].astype(float).quantile(0.90)

    # KL
    nat_kl_max = nat_seed["kl"].astype(float).quantile(0.90)
    loc_kl_min = loc_seed["kl"].astype(float).quantile(0.10)

    # Sig DMA (optional)
    nat_sig_min = nat_seed["sig_dma_count"].astype(float).quantile(0.1)
    loc_sig_max = loc_seed["sig_dma_count"].astype(float).quantile(0.9)


    nat_cond = {
        'cov': round(nat_cov_min, 3),
        'ent': round(nat_ent_min, 3),
        'kl': round(nat_kl_max, 3),
        'sig': round(nat_sig_min, 3)
    }
    loc_cond = {
        'cov': round(loc_cov_max, 3),
        'ent': round(loc_ent_max, 3),
        'kl': round(loc_kl_min, 3),
        'sig': round(loc_sig_max, 3)
    }
    return nat_cond, loc_cond


In [0]:
def print_suggested_thresh(nat_cond, loc_cond):
    print("Suggested thresholds (based on seeds):")
    print(f"  nat_cov_min ≈ {nat_cond['cov']:.3f}")
    print(f"  loc_cov_max ≈ {loc_cond['cov']:.3f}")
    print(f"  nat_ent_min ≈ {nat_cond['ent']:.3f}")
    print(f"  loc_ent_max ≈ {loc_cond['ent']:.3f}")
    print(f"  nat_kl_max  ≈ {nat_cond['kl']:.3f}")
    print(f"  loc_kl_min  ≈ {loc_cond['kl']:.3f}")
    print(f"  nat_sig_min ≈ {nat_cond['sig']:.3f}")
    print(f"  loc_sig_max ≈ {loc_cond['sig']:.3f}")

In [0]:
nat_cond, loc_cond = propose_thresholds_from_seeds(nat, loc)

In [0]:
print_suggested_thresh(nat_cond, loc_cond)

In [0]:
def classify_with_thresholds(row, nat_cond, loc_cond, vote_k=2):
    cov = float(row["coverage_score"])
    ent = float(row["entropy_norm"])
    kl = float(row["kl"])
    sig = float(row["sig_dma_count"])

    nat_score = sum(
        [
            cov >= nat_cond["cov"],
            ent >= nat_cond["ent"],
            kl <= nat_cond["kl"],
            sig >= nat_cond["sig"],
        ]
    )

    loc_score = sum(
        [
            cov <= loc_cond["cov"],
            ent <= loc_cond["ent"],
            kl >= loc_cond["kl"],
            sig <= loc_cond["sig"],
        ]
    )

    # strong decisions
    if nat_score >= vote_k and loc_score < vote_k:
        return "National"
    if loc_score >= vote_k and nat_score < vote_k:
        return "Local"

    # tie-breaker: pick larger score, then coverage
    if nat_score != loc_score:
        return "National" if nat_score > loc_score else "Local"

    # coverage tie-breaker (data-driven mid-zone)
    return "National" if cov >= (nat_cond["cov"] + loc_cond["cov"]) / 2 else "Local"

In [0]:
def propose_thresholds_from_labels(nat_df, loc_df, q_nat=0.10, q_loc=0.90):
    # National mins (lower tail) and max for KL (upper tail)
    nat_cond = {
        "cov": float(nat_df["coverage_score"].quantile(q_nat)),
        "ent": float(nat_df["entropy_norm"].quantile(q_nat)),
        "kl": float(nat_df["kl"].quantile(q_loc)),
        "sig": float(nat_df["sig_dma_count"].quantile(q_nat)),
    }

    # Local maxs (upper tail) and min for KL (lower tail)
    loc_cond = {
        "cov": float(loc_df["coverage_score"].quantile(q_loc)),
        "ent": float(loc_df["entropy_norm"].quantile(q_loc)),
        "kl": float(loc_df["kl"].quantile(q_nat)),
        "sig": float(loc_df["sig_dma_count"].quantile(q_loc)),
    }

    # rounding for readability only
    nat_cond = {k: round(v, 6) for k, v in nat_cond.items()}
    loc_cond = {k: round(v, 6) for k, v in loc_cond.items()}
    return nat_cond, loc_cond

In [0]:
def fit_thresholds_until_converged(
    df,
    init_nat_cond,
    init_loc_cond,
    q_nat=0.10,
    q_loc=0.90,
    vote_k=2,
    epsilon=1e-4,
    max_iters=30,
    max_label_change_rate=0.001,  # 0.1%
    verbose=True,
):
    df = df.copy()

    nat_cond = dict(init_nat_cond)
    loc_cond = dict(init_loc_cond)

    prev_labels = None

    def cond_delta(a, b):
        return max(abs(a[k] - b[k]) for k in a.keys())

    for it in range(1, max_iters + 1):
        # 1) label with current thresholds
        df["ad_type_tmp"] = df.apply(
            lambda r: classify_with_thresholds(r, nat_cond, loc_cond, vote_k), axis=1
        )

        # 2) compute label change rate
        if prev_labels is None:
            label_change_rate = 1.0
        else:
            label_change_rate = (df["ad_type_tmp"] != prev_labels).mean()

        prev_labels = df["ad_type_tmp"].copy()

        # 3) recompute thresholds from new labels
        nat_df = df[df["ad_type_tmp"] == "National"]
        loc_df = df[df["ad_type_tmp"] == "Local"]

        # Guardrails: if one side collapses, stop
        if len(nat_df) < 100 or len(loc_df) < 100:
            if verbose:
                print(
                    f"[iter {it}] STOP: one class too small (Nat={len(nat_df)}, Loc={len(loc_df)})"
                )
            break

        new_nat_cond, new_loc_cond = propose_thresholds_from_labels(
            nat_df, loc_df, q_nat=q_nat, q_loc=q_loc
        )

        # 4) check convergence
        d_nat = cond_delta(nat_cond, new_nat_cond)
        d_loc = cond_delta(loc_cond, new_loc_cond)

        if verbose:
            print(
                f"[iter {it}] Nat={len(nat_df)} Loc={len(loc_df)} | "
                f"Δnat={d_nat:.6f} Δloc={d_loc:.6f} | label_change={label_change_rate:.4%}\n"
                f"  nat_cond={new_nat_cond}\n"
                f"  loc_cond={new_loc_cond}"
            )

        if (
            d_nat < epsilon
            and d_loc < epsilon
            and label_change_rate < max_label_change_rate
        ):
            nat_cond, loc_cond = new_nat_cond, new_loc_cond
            if verbose:
                print(f"\n✅ Converged at iter {it}")
            break

        nat_cond, loc_cond = new_nat_cond, new_loc_cond

    df = df.rename(columns={"ad_type_tmp": "ad_type_final"})
    return df, nat_cond, loc_cond

In [0]:
def classify_binary_final(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    kl = row["kl"]
    sig = row["sig_dma_count"]

    # ----- NATIONAL CONDITIONS -----
    nat_conditions = [
        cov >= 0.045,
        ent >= 0.78,
        kl <= 2.75,
        sig >= 6.00
    ]
    nat_score = sum(nat_conditions)

    # ----- LOCAL CONDITIONS -----
    loc_conditions = [
        cov <= 0.02,
        ent <= 0.72,
        kl >= 3.10,
        sig <= 3.00
    ]
    loc_score = sum(loc_conditions)

    # ----- DECISION (NO MIXED) -----
    # Strong votes
    if ent >= 0.80:
        return "National"
    elif ent <= 0.60:
        return "Local"
    elif sig >= 6:
        return "National"
    elif sig <= 2:
        return "Local"
    elif cov <= 0.01:
        return "Local"
    elif cov >= 0.07:
        return "National"
    elif nat_score >= 2 and loc_score < 2:
        return "National"
    elif loc_score >= 2 and nat_score < 2:
        return "Local"
    elif nat_score >= loc_score:
        return "National"
    elif loc_score >= nat_score:
        return "Local"
    # Final Check
    else:
        return "Local"

In [0]:
new_df["ad_type_final"] = new_df.apply(classify_binary_final, axis=1)

nat = new_df[new_df["ad_type_final"] == "National"]
loc = new_df[new_df["ad_type_final"] == "Local"]
print(f'national Ads: {len(nat)}; Local Ads: {len(loc)}')
print_suggested_thresh(nat_cond, loc_cond)

In [0]:
def classify_binary_final(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    kl = row["kl"]
    sig = row["sig_dma_count"]

    # ----- NATIONAL CONDITIONS -----
    nat_conditions = [
        cov >= nat_cond["cov"],
        ent >= nat_cond["ent"],
        kl <= nat_cond["kl"],
        sig >= nat_cond["sig"],
    ]
    nat_score = sum(nat_conditions)

    # ----- LOCAL CONDITIONS -----
    loc_conditions = [
        cov <= loc_cond["cov"],
        ent <= loc_cond["ent"],
        kl >= loc_cond["kl"],
        sig <= loc_cond["sig"],
    ]
    loc_score = sum(loc_conditions)

    # ----- DECISION (NO MIXED) -----
    # Strong votes
    if ent >= 0.90:
        return "National"
    elif ent <= 0.40:
        return "Local"
    elif sig >= 10:
        return "National"
    elif sig <= 1:
        return "Local"
    elif cov <= 0.001:
        return "Local"
    elif cov >= 0.20:
        return "National"
    elif nat_score >= 2 and loc_score < 2:
        return "National"
    elif loc_score >= 2 and nat_score < 2:
        return "Local"
    elif nat_score >= loc_score:
        return "National"
    elif loc_score >= nat_score:
        return "Local"
    # Final Check
    else:
        return "Local"

In [0]:

new_df["ad_type_final"] = new_df.apply(classify_binary_final, axis=1)

nat = new_df[new_df["ad_type_final"] == "National"]
loc = new_df[new_df["ad_type_final"] == "Local"]
nat_cond, loc_cond = propose_thresholds_from_seeds(nat, loc)
print(f'national Ads: {len(nat)}; Local Ads: {len(loc)}')
print_suggested_thresh(nat_cond, loc_cond)

new_df["ad_type_final"] = new_df.apply(classify_binary_final, axis=1)

nat = new_df[new_df["ad_type_final"] == "National"]
loc = new_df[new_df["ad_type_final"] == "Local"]
nat_cond, loc_cond = propose_thresholds_from_seeds(nat, loc)
print(f'national Ads: {len(nat)}; Local Ads: {len(loc)}')
print_suggested_thresh(nat_cond, loc_cond)

new_df["ad_type_final"] = new_df.apply(classify_binary_final, axis=1)

nat = new_df[new_df["ad_type_final"] == "National"]
loc = new_df[new_df["ad_type_final"] == "Local"]
nat_cond, loc_cond = propose_thresholds_from_seeds(nat, loc)
print(f'national Ads: {len(nat)}; Local Ads: {len(loc)}')
print_suggested_thresh(nat_cond, loc_cond)

new_df["ad_type_final"] = new_df.apply(classify_binary_final, axis=1)

nat = new_df[new_df["ad_type_final"] == "National"]
loc = new_df[new_df["ad_type_final"] == "Local"]
nat_cond, loc_cond = propose_thresholds_from_seeds(nat, loc)
print(f'national Ads: {len(nat)}; Local Ads: {len(loc)}')
print_suggested_thresh(nat_cond, loc_cond)

new_df["ad_type_final"] = new_df.apply(classify_binary_final, axis=1)

nat = new_df[new_df["ad_type_final"] == "National"]
loc = new_df[new_df["ad_type_final"] == "Local"]
nat_cond, loc_cond = propose_thresholds_from_seeds(nat, loc)
print(f'national Ads: {len(nat)}; Local Ads: {len(loc)}')
print_suggested_thresh(nat_cond, loc_cond)
summarize_quantiles(nat, "National")
summarize_quantiles(loc, "Local")

In [0]:
summarize_quantiles(nat, "National")
summarize_quantiles(loc, "Local")

In [0]:
propose_thresholds_from_seeds(nat, loc)

In [0]:
import seaborn as sns

In [0]:
def plot_a_vs_b(df_plot, thresholds, a, b, a_name, b_name):
    plt.figure(figsize=(8, 7))

    for t in df_plot['ad_type_final'].unique():
        subset = df_plot[df_plot['ad_type_final'] == t]
        plt.scatter(
            subset[a],
            subset[b],
            alpha=0.25,
            label=t,
            s=2,
        )

    # plt.axvline(a_nat, linestyle='--')
    # plt.axvline(a_loc, linestyle='-.')
    # plt.axhline(b_nat, linestyle='--')
    # plt.axhline(b_loc, linestyle='-.')

    plt.ylabel(b_name)
    plt.xlabel(a_name)
    plt.title(f'{a_name} vs {b_name}')
    plt.legend()
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": [0.020, 0.057, "Coverage Score"],
    "entropy_norm": [0.794, 0.609, "Entropy Norm"],
    "kl": [2.96, 3.314, "KL"],
    "sig_dma_count": [5.528, 2.155, "Significant DMAs"],
    # "p_hat_pooled": ["nat_p_hat_cv_max", "loc_p_hat_cv_min", "P hat Pooled"],
    # "mix_ratio_station": [
    #     "nat_mix_ratio_tol",
    #     "loc_mix_ratio_min",
    #     "Station Mix Ratio",
    # ],
    # "competitive_pressure": ["nat_comp_min", "loc_comp_max", "Competitive Pressure"],
}
for a, v1 in plot_dict.items():
    for b, v2 in plot_dict.items():
        if a != b:
            plot_a_vs_b(
                df_plot=new_df,
                thresholds=thresholds,
                a=a,
                b=b,
                # a_nat=v1[0],
                # a_loc=v1[1],
                # b_nat=v2[0],
                # b_loc=v2[1],
                a_name=v1[2],
                b_name=v2[2],
            )

In [0]:
def plot_kde(df: pd.DataFrame, x: str, y: str, x_name: str, y_name: str):
    sns.kdeplot(
        data=df,
        x=x,
        y=y,
        # hue='ad_type_bin',
        levels=100,
        thresh=0.02,
        fill=False,
        alpha=0.4,
    )

    plt.title(f"KDE Plot - {x_name} vs {y_name}")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "kl": "KL",
    "sig_dma_count": "Significant DMAs"
}
done = []
for x, v1 in plot_dict.items():
    for y, v2 in plot_dict.items():
        if x != y and sorted([x, y]) not in done:
            try:
                done.append(sorted([x, y]))
                plot_kde(df=new_df, x=x, y=y, x_name=v1, y_name=v2)
            except ValueError:
                print(f'failed {x} {y}' )

In [0]:
def plot_kde(df: pd.DataFrame, x: str, y: str, x_name: str, y_name: str):
    sns.kdeplot(
        data=df,
        x=x,
        y=y,
        hue='ad_type_final',
        levels=100,
        thresh=0.02,
        fill=False,
        alpha=0.4,
    )

    plt.title(f"KDE Plot - {x_name} vs {y_name}")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "kl": "KL",
    "sig_dma_count": "Significant DMAs"
}
done = []
for x, v1 in plot_dict.items():
    for y, v2 in plot_dict.items():
        if x != y and sorted([x, y]) not in done:
            try:
                done.append(sorted([x, y]))
                plot_kde(df=new_df, x=x, y=y, x_name=v1, y_name=v2)
            except ValueError:
                print(f'failed {x} {y}' )

In [0]:
def plot_boxplot_by_label(df, col, label_col="ad_type_final"):
    data = [
        df[df[label_col] == "Local"][col].dropna(),
        df[df[label_col] == "National"][col].dropna()
        ]
    labels = ["Local", "National"]

    plt.figure(figsize=(6, 5))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(f"{col} distribution by label")
    plt.ylabel(col)
    plt.xlabel("ad_type")
    plt.grid(True, axis="y")
    plt.show()

In [0]:
def plot_hist_by_label(df, col, label_col="ad_type_final", bins=50):
    plt.figure(figsize=(7, 5))
    for label in ["Local", "National"]:
        subset = df[df[label_col] == label][col].dropna()
        plt.hist(subset, bins=bins, alpha=0.5, label=label)

    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"{col} histogram by label")
    plt.legend()
    plt.grid(True)
    plt.show()

In [0]:
for col in ["coverage_score", "entropy_norm", "kl", "sig_dma_count"]:
    new_df[col] = new_df[col].astype(float)
    plot_boxplot_by_label(new_df, col)

In [0]:
for col in ["coverage_score", "entropy_norm", "kl", "sig_dma_count"]:
    plot_hist_by_label(new_df, col)